In [ ]:
# generalization.py
import os
import gc
import time
import glob
import warnings
import argparse
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, average_precision_score, brier_score_loss

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
warnings.filterwarnings("ignore")
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

from core_glaucogate import (
    ConvNeXtBaseModel, ConvNeXtProGlaucoGate, GlaucoGateTotalLoss, apply_synthetic_degradation
)
from step3_train_eval import set_seed, compute_ece, compute_selective_metrics

def preprocess_domain_invariant_fundus(img_rgb, target_size=768):
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    thr = max(12, int(np.percentile(gray, 8)))
    mask = (gray > thr).astype(np.uint8)
    coords = cv2.findNonZero(mask)
    if coords is not None and len(coords) > 100:
        x, y, w, h = cv2.boundingRect(coords)
        if w > 0.3 * img_rgb.shape[1] and h > 0.3 * img_rgb.shape[0]:
            img_rgb = img_rgb[y:y+h, x:x+w]

    h, w, _ = img_rgb.shape
    if abs(h - w) > 0.08 * max(h, w):
        side = max(h, w)
        pad_top = (side - h) // 2
        pad_bot = side - h - pad_top
        pad_left = (side - w) // 2
        pad_right = side - w - pad_left
        img_rgb = cv2.copyMakeBorder(img_rgb, pad_top, pad_bot, pad_left, pad_right, cv2.BORDER_CONSTANT, value=(0, 0, 0))

    img_resized = cv2.resize(img_rgb, (target_size, target_size), interpolation=cv2.INTER_AREA)

    lab = cv2.cvtColor(img_resized, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.2, tileGridSize=(8, 8))
    l_clahe = clahe.apply(l)
    a_norm = np.clip((a.astype(np.float32) - np.mean(a)) + 128.0, 0, 255).astype(np.uint8)
    b_norm = np.clip((b.astype(np.float32) - np.mean(b)) + 128.0, 0, 255).astype(np.uint8)
    img_clahe = cv2.cvtColor(cv2.merge([l_clahe, a_norm, b_norm]), cv2.COLOR_LAB2RGB)

    blur_bg = cv2.GaussianBlur(img_clahe, (0, 0), sigmaX=target_size // 16)
    shade_corr = cv2.addWeighted(img_clahe, 1.35, blur_bg, -0.35, 16)

    yy, xx = np.ogrid[:target_size, :target_size]
    center = target_size / 2.0
    fov_mask = (((xx - center)**2 + (yy - center)**2) <= (0.485 * target_size)**2).astype(np.uint8)[:, :, None]
    final_img = (0.75 * img_clahe + 0.25 * shade_corr).astype(np.uint8) * fov_mask
    return final_img

def detect_anatomical_onh_center(img_rgb_512):
    h, w, _ = img_rgb_512.shape
    rg = 0.55 * img_rgb_512[:, :, 0].astype(np.float32) + 0.45 * img_rgb_512[:, :, 1].astype(np.float32)
    smooth = cv2.GaussianBlur(rg, (41, 41), 11.0)

    yy, xx = np.meshgrid(np.linspace(-1.0, 1.0, h), np.linspace(-1.0, 1.0, w), indexing="ij")
    interior_mask = ((xx**2 + yy**2) < 0.52) & (np.abs(yy) < 0.45)
    score_map = smooth * interior_mask.astype(np.float32)

    flat_idx = int(np.argmax(score_map))
    py, px = divmod(flat_idx, w)
    cx = float((px / max(1, w - 1)) * 2.0 - 1.0)
    cy = float((py / max(1, h - 1)) * 2.0 - 1.0)
    return np.clip(cx, -0.60, 0.60), np.clip(cy, -0.45, 0.45), 0.16

class DomainInvariantFundusDataset(Dataset):
    def __init__(self, csv_path, split=None, global_size=512, highres_size=768,
                 is_train=False, deg_prob=0.25, fixed_deg_type="none", fixed_severity=0.0):
        df = pd.read_csv(csv_path)
        if split is not None and "split" in df.columns:
            df = df[df["split"] == split].reset_index(drop=True)
        self.df = df
        self.global_size = global_size
        self.highres_size = highres_size
        self.is_train = is_train
        self.deg_prob = deg_prob
        self.fixed_deg_type = fixed_deg_type
        self.fixed_severity = fixed_severity

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_bgr = cv2.imread(str(row["image_path"]))
        if img_bgr is None:
            img_rgb = np.zeros((self.highres_size, self.highres_size, 3), dtype=np.uint8)
        else:
            img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        img_highres = preprocess_domain_invariant_fundus(img_rgb, target_size=self.highres_size)

        if self.is_train:
            if np.random.rand() < 0.5:
                img_highres = cv2.flip(img_highres, 1)
            gamma = np.random.uniform(0.82, 1.20)
            img_highres = np.clip(255.0 * ((img_highres / 255.0) ** gamma), 0, 255).astype(np.uint8)

        img_512_preview = cv2.resize(img_highres, (self.global_size, self.global_size), interpolation=cv2.INTER_AREA)
        auto_cx, auto_cy, auto_r = detect_anatomical_onh_center(img_512_preview)

        gt_cx = row.get("od_cx", np.nan)
        gt_cy = row.get("od_cy", np.nan)
        gt_r = row.get("od_r", np.nan)
        if pd.isna(gt_cx) or pd.isna(gt_cy):
            gt_cx, gt_cy, gt_r = auto_cx, auto_cy, auto_r

        if self.fixed_deg_type != "none" and self.fixed_severity > 0:
            img_highres, gt_qmap, omega_mask = apply_synthetic_degradation(
                img_highres, deg_type=self.fixed_deg_type, severity=self.fixed_severity, disc_center=(gt_cx, gt_cy)
            )
        elif self.is_train and np.random.rand() < self.deg_prob:
            sev = np.random.uniform(0.35, 0.85)
            img_highres, gt_qmap, omega_mask = apply_synthetic_degradation(
                img_highres, deg_type="random", severity=sev, disc_center=(gt_cx, gt_cy)
            )
        else:
            gt_qmap = np.ones((self.highres_size, self.highres_size), dtype=np.float32)
            omega_mask = np.zeros((self.highres_size, self.highres_size), dtype=np.float32)

        img_global = cv2.resize(img_highres, (self.global_size, self.global_size), interpolation=cv2.INTER_AREA)
        gt_qmap_small = cv2.resize(gt_qmap, (16, 16), interpolation=cv2.INTER_AREA)
        omega_small = cv2.resize(omega_mask, (16, 16), interpolation=cv2.INTER_NEAREST)

        highres_t = torch.from_numpy(img_highres).permute(2, 0, 1).float() / 255.0
        global_t = torch.from_numpy(img_global).permute(2, 0, 1).float() / 255.0

        yy, xx = torch.meshgrid(torch.linspace(-1, 1, 32), torch.linspace(-1, 1, 32), indexing="ij")
        saliency_prior = torch.exp(-((xx - gt_cx)**2 + (yy - gt_cy)**2) / (2 * 0.18**2))
        saliency_prior = (saliency_prior + 1e-6) / (saliency_prior.sum() + 1e-4)

        mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        highres_t = (highres_t - mean) / std
        global_t = (global_t - mean) / std

        return {
            "image_id": str(row["image_id"]),
            "patient_id": str(row.get("patient_id", row["image_id"])),
            "global_img": global_t,
            "highres_img": highres_t,
            "label": torch.tensor(float(row["label"]), dtype=torch.float32),
            "pseudo_domain": torch.tensor(int(row.get("pseudo_domain", 0)), dtype=torch.long),
            "gt_qmap": torch.from_numpy(gt_qmap_small).unsqueeze(0).float(),
            "omega_mask": torch.from_numpy(omega_small).unsqueeze(0).float(),
            "saliency_prior": saliency_prior.float(),
            "od_gt": torch.tensor([float(gt_cx), float(gt_cy), float(gt_r)], dtype=torch.float32)
        }